# Chapter 6: Foundational models and Semantic IDs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kimfalk/modern-recommender-systems/blob/main/notebooks/chapter-06/example_notebook.ipynb)

This notebook introduces the basics of recommender systems.

In [1]:
# Install datasets library for HuggingFace
!pip install -q datasets
!pip install -q mlflow  


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from pathlib import Path
import sys

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "recsys").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Added to sys.path: {project_root}")

Added to sys.path: /Users/kfalk/private/repos/manning/modern-recommender-systems


In [3]:
from recsys.utils.colab import setup_colab_environment, get_data_path, check_gpu
# One-line setup for Colab users
setup_colab_environment()

# Check GPU availability
check_gpu()

Not running in Colab, skipping setup.
⚠ No GPU detected.


False

In [4]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from recsys.data import loaders
from recsys.semantic_ids.utils import prepare_data
from recsys.semantic_ids.training import run_pipeline_debug
from recsys.semantic_ids.evaluations import evaluate_semantic_ids

DATA_PATH = get_data_path()

In [5]:
print("Ready to build recommender systems!")

Ready to build recommender systems!


In [6]:
from recsys.data.loaders import load_movielens_descriptions

movies = load_movielens_descriptions(data_dir=DATA_PATH)

movies.head()

Downloaded and saved movie descriptions to /Users/kfalk/private/repos/manning/modern-recommender-systems/notebooks/data/movie_descriptions.csv
Loaded 9837 movie descriptions
Failed to download descriptions: 'title'
Returning empty DataFrame...


""


In [7]:
# Reload the loaders module to pick up latest changes
import importlib
import recsys.data.loaders
importlib.reload(recsys.data.loaders)

<module 'recsys.data.loaders' from '/Users/kfalk/private/repos/manning/modern-recommender-systems/recsys/data/loaders.py'>

In [8]:
# Load movie descriptions from HuggingFace
from recsys.data.loaders import load_movielens_descriptions

movies_descriptions = load_movielens_descriptions(data_dir=DATA_PATH)

Loaded 9837 movie descriptions from cache


In [9]:
# Check the structure and columns

movies_descriptions.rename(columns={'Overview': 'description',
                                    'Title': 'title',
                                    'Genre': 'genres'}, inplace=True)
print(f"Shape: {movies_descriptions.shape}")
print(f"\nColumns: {movies_descriptions.columns.tolist()}")
movies_descriptions.head()

Shape: (9837, 9)

Columns: ['Release_Date', 'title', 'description', 'Popularity', 'Vote_Count', 'Vote_Average', 'Original_Language', 'genres', 'Poster_Url']


,Release_Date,title,description,Popularity,Vote_Count,Vote_Average,Original_Language,genres,Poster_Url
0,2021-12-15,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,5083.954,8940,8.3,en,"Action, Adventure, Science Fiction",https://image.tmdb.org/t/p/original/1g0dhYtq4i...
1,2022-03-01,The Batman,"In his second year of fighting crime, Batman u...",3827.658,1151,8.1,en,"Crime, Mystery, Thriller",https://image.tmdb.org/t/p/original/74xTEgt7R3...
2,2022-02-25,No Exit,Stranded at a rest stop in the mountains durin...,2618.087,122,6.3,en,Thriller,https://image.tmdb.org/t/p/original/vDHsLnOWKl...
3,2021-11-24,Encanto,"The tale of an extraordinary family, the Madri...",2402.201,5076,7.7,en,"Animation, Comedy, Family, Fantasy",https://image.tmdb.org/t/p/original/4j0PNHkMr5...
4,2021-12-22,The King's Man,As a collection of history's worst tyrants and...,1895.511,1793,7.0,en,"Action, Adventure, Thriller, War",https://image.tmdb.org/t/p/original/aq4Pwv5Xeu...


In [10]:
movies_descriptions = movies_descriptions[~movies_descriptions['title'].isna()]
movies_descriptions.shape



(9828, 9)

In [11]:
codebook_sizes = [16, 32, 128]  # Tiny capacity: max 512 combinations for 9,837 movies
internal_dim = 512  # Match input dimension
usage_loss_weight = 2.0  # Very strong penalty for imbalanced usage
epochs = 500  # Fewer epochs for faster iteration

In [12]:
import mlflow
import mlflow.pytorch

# Set experiment name
mlflow.set_experiment("semantic-ids-experiments")


<Experiment: artifact_location='/Users/kfalk/work/manning/modern-recommender-systems/notebooks/chapter-06/mlruns/1', creation_time=1771765791832, experiment_id='1', last_update_time=1771765791832, lifecycle_stage='active', name='semantic-ids-experiments', tags={}, workspace='default'>

In [13]:
recreate_embeddings = True

In [14]:
# Reload the module to pick up the fixes
import importlib
import recsys.semantic_ids.semantic_ids_pipeline
importlib.reload(recsys.semantic_ids.semantic_ids_pipeline)
importlib.reload(recsys.semantic_ids.training)
importlib.reload(recsys.semantic_ids.rqvae)
importlib.reload(recsys.semantic_ids.vector_quantizer)
importlib.reload(recsys.semantic_ids.evaluations)

# Now test with the fixed code
from recsys.semantic_ids.semantic_ids_pipeline import SemanticIDPipeline

from recsys.semantic_ids.utils import prepare_data

with mlflow.start_run(run_name="aggressive-clustering"):
    
    # Log parameters
    mlflow.log_params({
        "codebook_sizes": codebook_sizes,
        "internal_dim": internal_dim,
        "usage_loss_weight": usage_loss_weight,
        "epochs": epochs,
        "variance_weight": 1.0  # if using variance regularization
    })
   
    pipeline = SemanticIDPipeline(codebook_sizes, 
                                internal_dim,
                                usage_loss_weight=usage_loss_weight)

    if recreate_embeddings:
        texts = prepare_data(movies_descriptions)
        print("\nEncoding texts...")
        embeddings = pipeline.text_encoder.encode(texts, show_progress_bar=True)
        recreate_embeddings = False
            
    data_tensor = pipeline.initialize_data_with_embeddings(embeddings)
    pipeline.train(data_tensor, epochs=epochs)

    df_enriched = pipeline.inference(movies_descriptions, data_tensor)
    evaluate_semantic_ids(pipeline, df_enriched, data_tensor, codebook_sizes)
    # Log evaluation metrics
    num_clusters = df_enriched['semantic_id'].nunique()
    avg_cluster_size = len(df_enriched) / num_clusters
    
    mlflow.log_metrics({
        "num_unique_clusters": num_clusters,
        "avg_cluster_size": avg_cluster_size,
        "total_items": len(df_enriched)
    })
    
    # Save and log the model
    import torch
    import os
    
    # Create artifacts directory if it doesn't exist
    os.makedirs("mlflow_artifacts", exist_ok=True)
    
    # Save model state dict
    model_path = "mlflow_artifacts/rqvae_model.pth"
    torch.save(pipeline.rqvae.state_dict(), model_path)
    
    # Log as artifact
    mlflow.log_artifact(model_path)

Loading BERT...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding Text...

Encoding texts...


Batches:   0%|          | 0/308 [00:00<?, ?it/s]

...

100%|█████████▉| 499/500 [13:10<00:01,  1.59s/it, loss=0.6624, recon=0.0016, vq=0.4870, div=0]


Epoch 500:
  Loss: 0.662355
  Recon: 0.001631
  VQ: 0.487045
  Unique semantic IDs: 6973/9828 (71.0%)


100%|██████████| 500/500 [13:10<00:00,  1.58s/it, loss=0.6624, recon=0.0016, vq=0.4870, div=0]

  Level 1: 13/16 (81.2%) | Perplexity: 12.6
  Level 2: 32/32 (100.0%) | Perplexity: 30.8
  Level 3: 128/128 (100.0%) | Perplexity: 97.8

Generating semantic IDs...
Generated 9828 semantic IDs
  Level 1: 13 unique codes
  Level 2: 32 unique codes
  Level 3: 128 unique codes


SEMANTIC ID EVALUATION REPORT

1. RECONSTRUCTION QUALITY
------------------------------------------------------------
Reconstruction Metrics (Original Input Space):
  MSE: 0.0016 (lower is better, target < 0.1)
  Cosine Similarity: 0.6615 (higher is better, target > 0.9)
  Correlation: 0.6564 (target > 0.85)

2. CLUSTERING QUALITY
------------------------------------------------------------
Clustering Metrics (Top-Level Codes):
  Silhouette Score: -0.005 (range: -1 to 1, target > 0.3)
  Calinski-Harabasz: 2.6 (higher is better)

Clustering Metrics (Second-Level Codes):
  Silhouette Score: -0.053

3. SEMANTIC COHERENCE
------------------------------------------------------------
Debug: Star Wars ID: (9, 17, 1), Star Trek ID: (13, 4, 20), Matches: 0
✗ Star Wars                      vs Star Trek                     : 0/3 levels match (expected: high)

Passed: 0/1 tests

4. CODEBOOK UTILIZATION
------------------------------------------------------------

Level 1 (Codebook size: 16):
  Use

In [15]:
# Enhanced diagnostics - check loss components
import torch

print("="*70)
print("TRAINING DIAGNOSTICS")
print("="*70)

# 1. Check encoder output variance
with torch.no_grad():
    encoder_output = pipeline.rqvae.encoder(data_tensor[:100])
    print(f"\n1. Encoder Output:")
    print(f"   Std: {encoder_output.std().item():.4f}")
    print(f"   Mean: {encoder_output.mean().item():.4f}")
    print(f"   Min: {encoder_output.min().item():.4f}")
    print(f"   Max: {encoder_output.max().item():.4f}")

# 2. Check codebook usage and get actual loss values
print(f"\n2. Codebook Usage:")
for level_idx, vq in enumerate(pipeline.rqvae.quantizers):
    with torch.no_grad():
        if level_idx == 0:
            z = pipeline.rqvae.encoder(data_tensor)
        else:
            z = pipeline.rqvae.encoder(data_tensor)
            for prev_vq in pipeline.rqvae.quantizers[:level_idx]:
                z, _, _ = prev_vq(z)
        
        # Get loss components
        vq.train()
        quantized, vq_loss, indices = vq(z)
        
        unique_codes = torch.unique(indices).numel()
        
    print(f"\n   Level {level_idx+1} (size={codebook_sizes[level_idx]}):")
    print(f"   - Unique codes: {unique_codes}/{codebook_sizes[level_idx]} ({100*unique_codes/codebook_sizes[level_idx]:.1f}%)")
    print(f"   - VQ loss: {vq_loss.item():.4f}")
    print(f"   - Usage weight: {vq.usage_loss_weight}")

# 3. Check full forward pass
print(f"\n3. Full Forward Pass:")
with torch.no_grad():
    reconstructed, total_loss, codes = pipeline.rqvae(data_tensor[:100])
    recon_error = ((data_tensor[:100] - reconstructed) ** 2).mean()
    
    print(f"   - Reconstruction error: {recon_error.item():.6f}")
    print(f"   - Total VQ loss: {total_loss.item():.4f}")

print("\n" + "="*70)


TRAINING DIAGNOSTICS

1. Encoder Output:
   Std: 1.0857
   Mean: -0.0039
   Min: -3.6844
   Max: 4.0595

2. Codebook Usage:

   Level 1 (size=16):
   - Unique codes: 13/16 (81.2%)
   - VQ loss: 0.2755
   - Usage weight: 2.0

   Level 2 (size=32):
   - Unique codes: 9/32 (28.1%)
   - VQ loss: 1.5567
   - Usage weight: 2.0

   Level 3 (size=128):
   - Unique codes: 6/128 (4.7%)
   - VQ loss: 0.7031
   - Usage weight: 2.0

3. Full Forward Pass:
   - Reconstruction error: 0.001593
   - Total VQ loss: 0.4749



In [16]:
df_enriched.head()


,Release_Date,title,description,Popularity,Vote_Count,Vote_Average,Original_Language,genres,Poster_Url,semantic_id,leaf_id,final_id
4725,2013-02-01,Journey to the West: Conquering the Demons,In a world plagued by demons who cause great h...,21.870,248,7.0,zh,"Action, Fantasy, Adventure, Comedy",https://image.tmdb.org/t/p/original/XcFMQQfx1g...,"(1, 0, 111)",0,"(1, 0, 111, 0)"
9664,2019-04-05,The Head Hunter,"On the outskirts of a kingdom, a quiet but fie...",13.492,214,5.4,en,"Horror, Fantasy",https://image.tmdb.org/t/p/original/ol0DSLOIN8...,"(1, 0, 122)",0,"(1, 0, 122, 0)"
9520,2008-12-30,Jack Hunter and the Quest for Akhenaten's Tomb,Adventurer Jack Hunter sets off in pursuit of ...,13.631,22,6.4,en,Adventure,https://image.tmdb.org/t/p/original/AkjACojVQ8...,"(1, 0, 2)",0,"(1, 0, 2, 0)"
5494,1961-05-01,The Curse of the Werewolf,A child conceived after a demented beggar rape...,19.640,116,6.5,en,Horror,https://image.tmdb.org/t/p/original/nwtxrG9jd5...,"(1, 0, 21)",0,"(1, 0, 21, 0)"
815,2021-05-05,Those Who Wish Me Dead,A young boy finds himself pursued by two assas...,75.887,1089,6.8,en,Thriller,https://image.tmdb.org/t/p/original/xCEg6KowNI...,"(1, 0, 34)",0,"(1, 0, 34, 0)"


In [17]:

importlib.reload(recsys.semantic_ids.evaluations)
from matplotlib.pyplot import title
from recsys.semantic_ids.evaluations import test_semantic_coherence

# test_semantic_coherence(df_enriched)
def get_row(title, df):
    return df[df['title'] == title]['semantic_id'].values[0]
get_row("Star Wars", df_enriched)


(9, 17, 1)

In [18]:
df_enriched['title'].str.startswith('Star Wars')


4725    False
9664    False
9520    False
5494    False
815     False
        ...  
3499    False
8023    False
245     False
6075    False
702     False
Name: title, Length: 9828, dtype: bool

In [19]:
df_enriched = df_enriched[df_enriched.isna() == False]

df_enriched[df_enriched['title'].isna()]



,Release_Date,title,description,Popularity,Vote_Count,Vote_Average,Original_Language,genres,Poster_Url,semantic_id,leaf_id,final_id


In [20]:
df_enriched[df_enriched['title'].str.startswith('Star Wars')]


,Release_Date,title,description,Popularity,Vote_Count,Vote_Average,Original_Language,genres,Poster_Url,semantic_id,leaf_id,final_id
2631,2008-08-05,Star Wars: The Clone Wars,"Set between Episode II and III, The Clone Wars...",33.496,1532,6.1,en,"Animation, Action, Science Fiction, Adventure",https://image.tmdb.org/t/p/original/ywRtBu88SL...,"(2, 4, 89)",0,"(2, 4, 89, 0)"
1725,1999-05-19,Star Wars: Episode I - The Phantom Menace,"Anakin Skywalker, a young slave strong with th...",45.562,11924,6.5,en,"Adventure, Action, Science Fiction",https://image.tmdb.org/t/p/original/6wkfovpn7E...,"(9, 1, 94)",3,"(9, 1, 94, 3)"
1615,2002-05-15,Star Wars: Episode II - Attack of the Clones,Following an assassination attempt on Senator ...,47.917,10781,6.5,en,"Adventure, Action, Science Fiction",https://image.tmdb.org/t/p/original/oZNPzxqM2s...,"(9, 1, 94)",4,"(9, 1, 94, 4)"
1926,2005-05-17,Star Wars: Episode III - Revenge of the Sith,The evil Darth Sidious enacts his final plan f...,41.997,11189,7.4,en,"Science Fiction, Adventure, Action",https://image.tmdb.org/t/p/original/xfSAoBEm9M...,"(9, 1, 94)",5,"(9, 1, 94, 5)"
785,2015-12-15,Star Wars: The Force Awakens,Thirty years after defeating the Galactic Empi...,77.811,16772,7.3,en,"Action, Adventure, Science Fiction, Fantasy",https://image.tmdb.org/t/p/original/wqnLdwVXoB...,"(9, 1, 94)",6,"(9, 1, 94, 6)"
847,2017-12-13,Star Wars: The Last Jedi,Rey develops her newly discovered abilities wi...,73.522,12808,6.9,en,"Science Fiction, Action, Adventure",https://image.tmdb.org/t/p/original/kOVEVeg59E...,"(9, 1, 94)",7,"(9, 1, 94, 7)"
748,2019-12-18,Star Wars: The Rise of Skywalker,The surviving Resistance faces the First Order...,79.730,7588,6.4,en,"Action, Adventure, Science Fiction",https://image.tmdb.org/t/p/original/db32LaOibw...,"(9, 1, 94)",8,"(9, 1, 94, 8)"
625,1977-05-25,Star Wars,Princess Leia is captured and held hostage by ...,90.969,16952,8.2,en,"Adventure, Action, Science Fiction",https://image.tmdb.org/t/p/original/6FfCtAuVAW...,"(9, 17, 1)",0,"(9, 17, 1, 0)"


In [21]:

def investigate_cluster(df):# Check what's in the largest cluster
    largest_cluster_id = df['semantic_id'].value_counts().head(1).index[0]
    largest_cluster = df[df['semantic_id'] == largest_cluster_id]

    print(f"Items in cluster {largest_cluster_id}: {len(largest_cluster)}")
    print("\nSample titles:")
    print(largest_cluster[['title', 'genres']].head(20))

    print("\nGenre distribution:")
    
    print(largest_cluster['genres'].value_counts().head(10))

investigate_cluster(df_enriched)


Items in cluster (10, 30, 20): 39

Sample titles:
                                        title  \
2033                        Aloha Scooby-Doo!   
8624                      Big Top Scooby-Doo!   
3664                   Chill Out, Scooby-Doo!   
954              Happy Halloween, Scooby-Doo!   
7655      LEGO Scooby-Doo! Blowout Beach Bash   
891                                    Scoob!   
443                                Scooby-Doo   
1674         Scooby-Doo 2: Monsters Unleashed   
7232          Scooby-Doo and the Ghoul School   
3306              Scooby-Doo on Zombie Island   
3901              Scooby-Doo! Abracadabra-Doo   
6209  Scooby-Doo! Adventures: The Mystery Map   
2665                   Scooby-Doo! Camp Scare   
2213    Scooby-Doo! Curse of the Lake Monster   
5921                Scooby-Doo! Frankencreepy   
4991     Scooby-Doo! Legend of the Phantosaur   
5988      Scooby-Doo! Mask of the Blue Falcon   
7690       Scooby-Doo! Meets the Boo Brothers   
6529         Scooby